<a href="https://colab.research.google.com/github/mehmoodanas/Bootcamp/blob/main/Day-3/Day_3_Tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Load dataset

In [9]:
# Importing the required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Import the dataset
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

df.head()


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


#Step 2: Train Decision Tree, Random Forest and Gradient Boosting Model


##Decision Tree

In [21]:
# Putting feature variables into X and target variable into y

# Feature variables
X = df.drop('target', axis=1)

# Target variable
y = df['target']


# Train-Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(398, 30) (171, 30)


In [22]:
# Import DecisionTreeClassifier and fit the data

from sklearn.tree import DecisionTreeClassifier

classifier_dt = DecisionTreeClassifier(
    random_state=42
)

# Train the model
classifier_dt.fit(X_train, y_train)

# Make predictions on the test set
y_pred_dt = classifier_dt.predict(X_test)

from sklearn.metrics import accuracy_score

# Accuracy
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("Accuracy is:", accuracy_dt)

Accuracy is: 0.9181286549707602


##Random Forest

In [23]:
# Import RandomForestClassifier and fit the data

from sklearn.ensemble import RandomForestClassifier

classifier_rf = RandomForestClassifier(
    n_estimators=10,
    random_state=42
)

# Train the model
classifier_rf.fit(X_train, y_train)

# Make predictions on the test set
y_pred_rf = classifier_rf.predict(X_test)

# Accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("Accuracy is:", accuracy_rf)

Accuracy is: 0.9298245614035088


## Gradient Boosting

In [24]:
# Import GradientBoostingClassifier and fit the data

from sklearn.ensemble import GradientBoostingClassifier

classifier_gb = GradientBoostingClassifier(
    random_state=42
)

# Train the model
classifier_gb.fit(X_train, y_train)

# Make predictions on the test set
y_pred_gb = classifier_gb.predict(X_test)

# Accuracy
accuracy_gb = accuracy_score(y_test, y_pred_gb)
print("Accuracy is:", accuracy_gb)

Accuracy is: 0.9473684210526315


#Step 3: Tune Random Forest hyperparameters

In [27]:
# Hyperparameter tuning for Random Forest using GridSearchCV

from sklearn.model_selection import GridSearchCV

# Base model
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# Parameter grid
params = {
    'max_depth': [2, 3, 5],
    'min_samples_leaf': [5, 10, 15, 18],
    'n_estimators': [10, 25, 30, 40]
}

# Instantiate GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=params,
    cv=4,
    n_jobs=-1,
    verbose=1,
    scoring='accuracy'
)

# Fit on training data
grid_search.fit(X_train, y_train)

# Best results
print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

# Evaluate best model on test data
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)
print("Accuracy:", accuracy_best)

Fitting 4 folds for each of 48 candidates, totalling 192 fits
Best Parameters: {'max_depth': 5, 'min_samples_leaf': 5, 'n_estimators': 30}
Best CV Accuracy: 0.9623737373737373
Accuracy: 0.9298245614035088


#Step 4: Compare ROC-AUC and F1 for all models

In [29]:
from sklearn.metrics import f1_score, roc_auc_score

# All trained models together
models = {
    "Decision Tree": classifier_dt,
    "Random Forest": classifier_rf,
    "Gradient Boosting": classifier_gb,
    "Tuned Random Forest": best_model
}

# Compare the models on the same test set
for name, model in models.items():
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, probabilities)

    print(f"{name}: Accuracy = {accuracy:.3f}, F1 = {f1:.3f}, ROC-AUC = {roc_auc:.3f}")

Decision Tree: Accuracy = 0.918, F1 = 0.935, ROC-AUC = 0.913
Random Forest: Accuracy = 0.930, F1 = 0.943, ROC-AUC = 0.982
Gradient Boosting: Accuracy = 0.947, F1 = 0.959, ROC-AUC = 0.990
Tuned Random Forest: Accuracy = 0.930, F1 = 0.944, ROC-AUC = 0.990
